# Creating the CSV train-val-test splits files
Make sure that the directories are all set well! 

In [ ]:

"""
make_splits_07_train_08_valtest.py

Fixed dataset split:
- TRAIN: all patterns from simulation 07
- VAL:   half of patterns from simulation 08
- TEST:  other half of patterns from simulation 08

Behavior:
- BASE_DIR is defined at the top of this file
- Output folder is BASE_DIR / "splits"
- The splits folder is created if it does not exist
- CSV files are NOT overwritten unless OVERWRITE = True
"""

from __future__ import annotations

import csv
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional

# CHANGE HERE FOR YOUR OWN BASEDIR

BASE_DIR = Path(# SET HERE)  
SEED = 123
OVERWRITE = True  # set True to regenerate CSVs



@dataclass(frozen=True)
class PatternItem:
    sim_id: str
    pattern_id: str
    pattern_dir: Path
    gt_path: Path
    noisy_dir: Path
    sofi_run_dir: Path
    sofi_c2_files: List[Path]
    sofi_ref_files: List[Path]


def _is_pattern_dir(p: Path) -> bool:
    return p.is_dir() and p.name.endswith("_Training_")


def _find_sim_dir(base: Path, sim_id: str) -> Path:
    matches = [
        p for p in base.iterdir()
        if p.is_dir() and p.name.split("_", 1)[0] == sim_id
    ]
    if not matches:
        raise FileNotFoundError(f"No simulation folder found for sim_id='{sim_id}' in {base}")
    if len(matches) > 1:
        matches.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return matches[0]


def _pick_sofi_run_dir(sofi_root: Path, prefer_contains: str = "1_100f") -> Optional[Path]:
    if not sofi_root.exists():
        return None

    candidates = [d for d in sofi_root.iterdir() if d.is_dir()]
    if not candidates:
        return None

    preferred = [d for d in candidates if prefer_contains in d.name]
    if preferred:
        preferred.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        return preferred[0]

    candidates.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]


def _collect_pattern_items(sim_dir: Path) -> List[PatternItem]:
    sim_id = sim_dir.name.split("_", 1)[0]
    items: List[PatternItem] = []

    for pat_dir in sorted(sim_dir.iterdir()):
        if not _is_pattern_dir(pat_dir):
            continue

        pattern_id = pat_dir.name.split("_", 1)[0]

        gt_path = pat_dir / "GT" / "ConvDownUp" / "Rescaled_GT" / "2o_Training_GTconvres_135nm.tif"
        noisy_dir = pat_dir / "Noisy"

        sofi_root = pat_dir / "_SOFI_Results_fwhm2.7"
        sofi_run_dir = _pick_sofi_run_dir(sofi_root) or Path("")

        sofi_c2_files: List[Path] = []
        sofi_ref_files: List[Path] = []

        if sofi_run_dir and sofi_run_dir.exists():
            for f in sorted(sofi_run_dir.iterdir()):
                if not f.is_file():
                    continue
                name = f.name.lower()
                if name.endswith("_noisyvid__mctsofi_c2.tif") and not name.startswith("04_"):
                    sofi_c2_files.append(f)

        items.append(
            PatternItem(
                sim_id=sim_id,
                pattern_id=pattern_id,
                pattern_dir=pat_dir,
                gt_path=gt_path,
                noisy_dir=noisy_dir,
                sofi_run_dir=sofi_run_dir,
                sofi_c2_files=sofi_c2_files,
                sofi_ref_files=sofi_ref_files,
            )
        )

    return items


def _write_csv(path: Path, items: List[PatternItem]) -> None:
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(
            [
                "sim_id",
                "pattern_id",
                "pattern_dir",
                "gt_path",
                "noisy_dir",
                "sofi_run_dir",
                "sofi_c2_files",
                "sofi_ref_files",
            ]
        )
        for it in items:
            w.writerow(
                [
                    it.sim_id,
                    it.pattern_id,
                    str(it.pattern_dir),
                    str(it.gt_path),
                    str(it.noisy_dir),
                    str(it.sofi_run_dir),
                    ";".join(str(x) for x in it.sofi_c2_files),
                    ";".join(str(x) for x in it.sofi_ref_files),
                ]
            )


def main() -> None:
    if not BASE_DIR.exists():
        raise FileNotFoundError(f"BASE_DIR does not exist: {BASE_DIR}")

    out_dir = BASE_DIR / "splits"
    out_dir.mkdir(parents=True, exist_ok=True)

    train_csv = out_dir / "train.csv"
    val_csv = out_dir / "val.csv"
    test_csv = out_dir / "test.csv"

    if not OVERWRITE and (train_csv.exists() or val_csv.exists() or test_csv.exists()):
        print(f"Splits already exist in: {out_dir}")
        print("Existing files:")
        for p in [train_csv, val_csv, test_csv]:
            if p.exists():
                print(" -", p.name)
        print("Set OVERWRITE = True to regenerate.")
        return

    # Fixed split rule
    sim07_dir = _find_sim_dir(BASE_DIR, "07")
    sim08_dir = _find_sim_dir(BASE_DIR, "08")

    train_items = _collect_pattern_items(sim07_dir)
    sim08_items = _collect_pattern_items(sim08_dir)

    if len(sim08_items) < 2:
        raise RuntimeError("Need at least 2 patterns in sim 08 to split val/test.")

    rng = random.Random(SEED)
    rng.shuffle(sim08_items)

    half = len(sim08_items) // 2
    val_items = sim08_items[:half]
    test_items = sim08_items[half:]

    # Safety: no pattern leakage
    def dirs(xs: List[PatternItem]) -> set[str]:
        return {str(x.pattern_dir) for x in xs}

    if dirs(train_items) & dirs(val_items) or dirs(train_items) & dirs(test_items) or dirs(val_items) & dirs(test_items):
        raise RuntimeError("Pattern leakage detected (this should never happen).")

    _write_csv(train_csv, train_items)
    _write_csv(val_csv, val_items)
    _write_csv(test_csv, test_items)

    print("Splits written to:", out_dir)
    print(f"Train patterns (07): {len(train_items)}")
    print(f"Val patterns   (08): {len(val_items)}")
    print(f"Test patterns  (08): {len(test_items)}")


if __name__ == "__main__":
    main()


# Creating the preprocessed folders
It's important the SN2N_resampling_fourier_RL_deconvolution_py is able to be imported, for which you need it to be in the same directory or file

In [ ]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import csv
from pathlib import Path
from typing import Dict, List

# CONFIGURATIONS
BASE_DIR = Path(r"SET HERE")  # <-- change once
SPLITS_DIR = Path(r"SET HERE") / "splits"
OUT_DIR = SPLITS_DIR / "preprocessed-no-p2p"                 # <-- new folder under splits/
OVERWRITE = False                                     # set True to regenerate
VERBOSE = True

#  adjust this import to match your preprocessing script filename/module.
from SN2N_resampling_fourier_RL_deconvolution_py import run_pipeline, make_sn2n_sample, save_sn2n_sample


def _parse_semicolon_list(s: str) -> List[str]:
    s = (s or "").strip()
    if not s:
        return []
    return [x.strip() for x in s.split(";") if x.strip()]


def _read_split_csv(csv_path: Path) -> List[Dict[str, str]]:
    with csv_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return list(reader)


def process_split(split_name: str) -> None:
    split_csv = SPLITS_DIR / f"{split_name}.csv"
    if not split_csv.exists():
        raise FileNotFoundError(f"Missing split file: {split_csv}")

    rows = _read_split_csv(split_csv)
    if VERBOSE:
        print(f"\n[{split_name}] Loaded {len(rows)} pattern-rows from {split_csv}")

    out_split_dir = OUT_DIR / split_name
    out_split_dir.mkdir(parents=True, exist_ok=True)

    written = 0
    skipped = 0
    missing = 0

    for r in rows:
        sim_id = r.get("sim_id", "NA")
        pattern_id = r.get("pattern_id", "NA")
        c2_list = _parse_semicolon_list(r.get("sofi_c2_files", ""))

        if not c2_list:
            # Some patterns may not have c2 files indexed; just skip
            continue

        for c2_path_str in c2_list:
            c2_path = Path(c2_path_str)
            if not c2_path.exists():
                missing += 1
                if VERBOSE:
                    print(f"  [MISSING] {c2_path}")
                continue

            # output filename: informative + stable
            # stem example: "07_noisyvid__mctsofi_c2"
            out_name = f"{sim_id}_{pattern_id}_{c2_path.stem}_SN2N.tif"
            out_path = out_split_dir / out_name

            if out_path.exists() and not OVERWRITE:
                skipped += 1
                continue

            # Run your preprocessing pipeline
            res = run_pipeline(str(c2_path))  # returns dict with left_rl/right_rl
            sample = make_sn2n_sample(res["left_rl"], res["right_rl"])  # (H, 2W)
            save_sn2n_sample(sample, out_path)

            written += 1
            if VERBOSE and written % 25 == 0:
                print(f"  wrote {written} samples so far...")

    print(
        f"[{split_name}] Done. wrote={written}, skipped={skipped}, missing_inputs={missing}\n"
        f"Outputs in: {out_split_dir}"
    )


def main() -> None:
    if not SPLITS_DIR.exists():
        raise FileNotFoundError(f"Splits directory not found: {SPLITS_DIR}")
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Create preprocessed/{train,val,test}
    for split in ("train", "val", "test"):
        process_split(split)


if __name__ == "__main__":
    main()



[train] Loaded 100 pattern-rows from C:\Users\ntpar\Downloads\Capstone\splits\train.csv
  wrote 25 samples so far...
  wrote 50 samples so far...
  wrote 75 samples so far...
  wrote 100 samples so far...
  wrote 125 samples so far...
  wrote 150 samples so far...
  wrote 175 samples so far...
  wrote 200 samples so far...
  wrote 225 samples so far...
  wrote 250 samples so far...
  wrote 275 samples so far...
  wrote 300 samples so far...
  wrote 325 samples so far...
  wrote 350 samples so far...
  wrote 375 samples so far...
  wrote 400 samples so far...
  wrote 425 samples so far...
  wrote 450 samples so far...
  wrote 475 samples so far...
  wrote 500 samples so far...
  wrote 525 samples so far...
  wrote 550 samples so far...
  wrote 575 samples so far...
  wrote 600 samples so far...
  wrote 625 samples so far...
  wrote 650 samples so far...
  wrote 675 samples so far...
  wrote 700 samples so far...
  wrote 725 samples so far...
  wrote 750 samples so far...
  wrote 775 sa